# 19 · Transformers and Large Language Models

- **Level:** Modern (2017 → present)
- **Typical semester:** Semester 7–8 electives (NLP, Generative AI) and self-study
- **Prerequisites:** 14, 16, 18
- **Matches code in:** `lipipy.dl` (transformer) and `lipipy.research` (LLM variants)


## 1. Scaled dot-product attention

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

$Q = XW_Q$, $K = XW_K$, $V = XW_V$. The causal mask $M_{ij} = -\infty$ for $j > i$ (decoder). Scaling by $\sqrt{d_k}$ keeps the logits' variance near 1 when $q, k$ have unit-variance entries.

Cost: $O(n^2d)$ time, $O(n^2)$ memory for the score matrix.


## 2. Multi-head attention and its variants

$$\text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i), \qquad \text{MHA}(X) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)W_O$$
with $d_k = d_{model}/h$.

**Multi-query attention (MQA):** all heads share one $K, V$. **Grouped-query attention (GQA):** $g$ groups of heads share $K, V$. Both shrink the KV cache by $h/g$.

**Multi-head latent attention (MLA):** compress keys and values into a low-rank latent $c = XW_{DKV}$, then $K = cW_{UK}$, $V = cW_{UV}$; cache only $c$.

**Cross-attention:** $Q$ from the decoder, $K, V$ from the encoder.


## 3. The transformer block

Pre-norm decoder block (GPT/LLaMA style):
$$h = x + \text{MHA}(\text{Norm}(x)), \qquad y = h + \text{FFN}(\text{Norm}(h))$$
$$\text{FFN}(x) = W_2\,\phi(W_1x) \quad\text{or}\quad \text{SwiGLU FFN}(x) = W_2(\text{SiLU}(W_1x)\odot W_3x)$$
FFN hidden size is usually $4d$ (or $\frac83d$ for SwiGLU to keep the parameter count equal).

Parameter count per block $\approx12d^2$ (4 attention projections $4d^2$ + FFN $8d^2$). A model with $L$ layers has about $12Ld^2$ non-embedding parameters.


## 4. Positional information

**Sinusoidal:** $PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$, $\;PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$

**Rotary position embedding (RoPE):** rotate each pair $(x_{2i}, x_{2i+1})$ by angle $m\theta_i$, $\theta_i = 10000^{-2i/d}$:
$$\begin{pmatrix}x'_{2i}\\x'_{2i+1}\end{pmatrix} = \begin{pmatrix}\cos m\theta_i & -\sin m\theta_i\\\sin m\theta_i & \cos m\theta_i\end{pmatrix}\begin{pmatrix}x_{2i}\\x_{2i+1}\end{pmatrix}, \qquad \langle R_mq, R_nk\rangle = \langle q, R_{n-m}k\rangle$$
Context extension: position interpolation $m\to m\frac{L}{L'}$; NTK-aware scaling of the base; YaRN.

**ALiBi:** add $-s_h\cdot|i - j|$ to attention logits, with a head-specific slope $s_h$.

**Relative position bias (T5):** learned bias $b_{\text{bucket}(i - j)}$.


## 5. Language-model objectives

**Causal LM:** $\mathcal L = -\sum_t\log p_\theta(x_t|x_{<t})$; perplexity $= \exp(\mathcal L/N)$

**Masked LM (BERT):** predict the 15% masked tokens, $-\sum_{t\in\mathcal M}\log p(x_t|x_{\setminus\mathcal M})$

**Span corruption (T5)**, **prefix LM**, **fill-in-the-middle**, **multi-token prediction:** $-\sum_t\sum_{k=1}^K\log p(x_{t+k}|x_{\le t})$

Logits $= hW_E^\top$ with tied embeddings; softmax over a vocabulary $V$ (typically 32k–256k).


## 6. Tokenization

**Byte-pair encoding:** repeatedly merge the most frequent adjacent pair. **WordPiece:** merge the pair maximizing $\frac{\text{count}(ab)}{\text{count}(a)\text{count}(b)}$. **Unigram LM:** keep the subword vocabulary that maximizes likelihood, pruning tokens with least loss.

Compression ratio (bytes per token) is about 3.5–4.5 for English text.


## 7. Compute, memory and inference

**Training FLOPs:** $C\approx6ND$ ($N$ parameters, $D$ tokens; $2N$ forward + $4N$ backward per token)

**Inference FLOPs:** $\approx2N$ per generated token (+ attention $\approx 2n_{layer}n_{ctx}d$)

**KV-cache size:** $2\times n_{layers}\times n_{kv\_heads}\times d_{head}\times\text{seq\_len}\times\text{batch}\times\text{bytes}$

**Arithmetic intensity** = FLOPs / bytes moved; decoding is memory-bandwidth bound, prefill is compute bound.

**Model FLOPs utilization:** MFU $= \frac{6ND/\text{time}}{\text{peak FLOP/s}}$


## 8. FlashAttention (online softmax)

Process keys in blocks. For each query row keep running max $m$, normalizer $\ell$ and output $o$:
$$m' = \max(m, \max_j s_j), \qquad \ell' = e^{m - m'}\ell + \sum_je^{s_j - m'}, \qquad o' = \frac{e^{m - m'}\ell\,o + \sum_je^{s_j - m'}v_j}{\ell'}$$
The result is exact, with $O(n)$ memory instead of $O(n^2)$.


## 9. Efficient attention variants

**Linear attention:** $\text{softmax}(QK^\top)V\approx\frac{\phi(Q)(\phi(K)^\top V)}{\phi(Q)\phi(K)^\top\mathbf 1}$, cost $O(nd^2)$; as a recurrence $S_t = S_{t-1} + \phi(k_t)v_t^\top$, $o_t = \frac{\phi(q_t)^\top S_t}{\phi(q_t)^\top z_t}$

Sliding-window attention (window $w$, cost $O(nw)$); sparse / block-sparse patterns; attention sinks; ring attention for very long contexts across devices.


## 10. Mixture of experts

$$y = \sum_{i\in\text{TopK}(g(x))}g_i(x)E_i(x), \qquad g(x) = \text{softmax}(W_gx)$$

**Load-balancing auxiliary loss (Switch):** $\mathcal L_{aux} = \alpha N\sum_{i=1}^Nf_iP_i$, where $f_i$ = fraction of tokens routed to expert $i$, $P_i$ = mean router probability for expert $i$.

**Router z-loss:** $\frac1B\sum\left(\log\sum_je^{z_j}\right)^2$. Capacity factor limits tokens per expert. Active parameters per token ≪ total parameters.


## 11. Scaling laws

**Kaplan et al. (2020):** $L(N)\approx\left(\frac{N_c}{N}\right)^{\alpha_N}$, $L(D)\approx\left(\frac{D_c}{D}\right)^{\alpha_D}$, with $\alpha_N\approx0.076$, $\alpha_D\approx0.095$

**Chinchilla (Hoffmann et al., 2022):** $L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$ with fitted $E\approx1.69$, $\alpha\approx0.34$, $\beta\approx0.28$. Compute-optimal training uses $N\propto C^{0.5}$, $D\propto C^{0.5}$, roughly **20 tokens per parameter**.

Inference-aware scaling: smaller models trained on many more tokens when serving cost matters. Test-time compute scaling: accuracy improves with more sampled reasoning or search at inference.


## 12. Decoding

Greedy $\arg\max_x p(x)$; beam search (keep top-$B$ partial sequences by $\sum\log p$, length penalty $\frac{1}{|y|^\alpha}$)

**Temperature:** $p_i\propto\exp(z_i/T)$. **Top-$k$:** keep the $k$ most likely tokens. **Top-$p$ (nucleus):** keep the smallest set with cumulative probability $\ge p$. **Min-$p$:** keep tokens with $p_i\ge p_{\min}\cdot\max_jp_j$.

Repetition penalty; contrastive decoding $\log p_{expert} - \log p_{amateur}$

**Speculative decoding:** a draft model proposes tokens; accept token $x$ with probability $\min\left(1, \frac{p(x)}{q(x)}\right)$, otherwise resample from $\text{norm}(\max(0, p - q))$. The output distribution exactly matches the target model.


## 13. Parameter-efficient fine-tuning

**LoRA:** $W = W_0 + \frac{\alpha}{r}BA$, $B\in\mathbb R^{d\times r}$ (init 0), $A\in\mathbb R^{r\times k}$, $r\ll\min(d,k)$; trainable params $r(d + k)$ instead of $dk$

**QLoRA:** LoRA on a 4-bit NormalFloat-quantized base model. **DoRA:** $W = m\frac{W_0 + BA}{\|W_0 + BA\|_c}$ (magnitude and direction)

**Prefix / prompt tuning:** learn virtual tokens prepended to keys and values. **Adapters:** $h + W_{up}\phi(W_{down}h)$


## 14. Quantization and compression of LLMs

Absmax int8: $q = \text{round}\left(127\frac{x}{\max|x|}\right)$. Per-channel / per-group scales. Outlier features handled by mixed precision (LLM.int8()).

**GPTQ:** quantize weights column by column, updating remaining weights with the inverse Hessian $H = 2XX^\top$ to compensate the error.

**AWQ:** scale salient channels (by activation magnitude) before quantization.

**BitNet b1.58:** ternary weights $W_q = \text{RoundClip}\left(\frac{W}{\text{mean}|W|}, -1, 1\right)$.


## 15. Retrieval-augmented generation

$p(y|x) = \sum_{z\in\text{TopK}(x)}p(z|x)p(y|x, z)$; dense retrieval score $s(q, d) = E_q(q)^\top E_d(d)$

**BM25:** $\sum_{t\in q}\text{IDF}(t)\frac{f(t,d)(k_1 + 1)}{f(t,d) + k_1\left(1 - b + b\frac{|d|}{\text{avgdl}}\right)}$, $\;\text{IDF}(t) = \ln\frac{N - n_t + 0.5}{n_t + 0.5}$ (smoothed variants add 1 inside the log)

**TF-IDF:** $\text{tf}(t,d)\cdot\log\frac{N}{\text{df}(t)}$

**Reciprocal rank fusion:** $\text{RRF}(d) = \sum_r\frac{1}{k + \text{rank}_r(d)}$, $k\approx60$


## 16. In-context learning (theory notes)

Linear self-attention can implement one step of gradient descent on an in-context regression problem. Induction heads (previous-token head + copying head) implement "[A][B] … [A] → [B]" pattern completion.
